# Notebook 8: Scalability Analysis
## Strong Scaling, Weak Scaling, and Performance Bottleneck Analysis

**Strong Scaling**: Same data size, increase cores → measure speedup  
**Weak Scaling**: Increase data + resources proportionally → measure efficiency  

This notebook separates **70% from 85% students** in assessment.

In [ ]:
# ============================================================================
# Cell 1: Setup
# ============================================================================
import os, sys, time, gc
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

sns.set_theme(style='whitegrid', palette='viridis', font_scale=1.1)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# ============================================================================
# Cell 2: Helper Functions
# ============================================================================
def create_spark(cores, memory='8g'):
    """Create a Spark session with specific core count."""
    spark = (
        SparkSession.builder
        .appName(f'ScalabilityTest_{cores}cores')
        .master(f'local[{cores}]')
        .config('spark.driver.memory', memory)
        .config('spark.executor.memory', '4g')
        .config('spark.driver.maxResultSize', '4g')
        .config('spark.sql.shuffle.partitions', max(cores * 4, 16))
        .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
        .config('spark.sql.adaptive.enabled', 'true')
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel('ERROR')
    return spark

def time_training(spark, data_path, fraction=1.0):
    """Load data, train LR, and return timing info."""
    # Load
    start_load = time.time()
    df = spark.read.parquet(data_path)
    if fraction < 1.0:
        df = df.sample(False, fraction, seed=42)
    df = df.withColumnRenamed('is_viral', 'label').cache()
    count = df.count()  # Force materialization
    load_time = time.time() - start_load
    
    # Split
    train, test = df.randomSplit([0.8, 0.2], seed=42)
    train.cache()
    
    # Train
    lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=50, regParam=0.01)
    start_train = time.time()
    model = lr.fit(train)
    train_time = time.time() - start_train
    
    # Evaluate
    preds = model.transform(test)
    evaluator = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC')
    auc = evaluator.evaluate(preds)
    
    # Cleanup
    df.unpersist()
    train.unpersist()
    
    return {
        'rows': count,
        'load_time': round(load_time, 2),
        'train_time': round(train_time, 2),
        'total_time': round(load_time + train_time, 2),
        'auc': round(auc, 4)
    }

print('Helper functions defined.')

## 1. Strong Scaling Analysis

**Definition**: Keep data size constant, increase the number of processing cores.  
**Expected**: Training time should decrease as cores increase (ideal = linear speedup).  
**Reality**: Sub-linear due to communication overhead and Amdahl's law.

In [ ]:
# ============================================================================
# Cell 3: Strong Scaling Test
# ============================================================================
PARQUET_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'train.parquet')
DATA_FRACTION = 0.5  # Use 50% of data for scaling tests (faster)

CORE_CONFIGS = [1, 2, 4, 8]  # Test these core counts

strong_scaling_results = []

print('STRONG SCALING TEST')
print('=' * 60)
print(f'Data fraction: {DATA_FRACTION*100:.0f}%')
print(f'Core configurations: {CORE_CONFIGS}')
print()

for cores in CORE_CONFIGS:
    print(f'Testing with {cores} core(s)...')
    
    # Stop existing session
    try:
        SparkSession.builder.getOrCreate().stop()
    except:
        pass
    time.sleep(2)
    
    spark = create_spark(cores)
    result = time_training(spark, PARQUET_PATH, fraction=DATA_FRACTION)
    result['cores'] = cores
    strong_scaling_results.append(result)
    
    print(f'  Rows: {result["rows"]:,}, Train: {result["train_time"]}s, '
          f'Total: {result["total_time"]}s, AUC: {result["auc"]}')
    
    spark.stop()
    gc.collect()

strong_df = pd.DataFrame(strong_scaling_results)

# Compute speedup
baseline_time = strong_df[strong_df['cores'] == 1]['train_time'].values[0]
strong_df['speedup'] = baseline_time / strong_df['train_time']
strong_df['efficiency'] = strong_df['speedup'] / strong_df['cores']
strong_df['ideal_speedup'] = strong_df['cores'].astype(float)

print('\nStrong Scaling Results:')
print(strong_df[['cores', 'train_time', 'speedup', 'ideal_speedup', 'efficiency', 'auc']].to_string(index=False))

In [ ]:
# ============================================================================
# Cell 4: Strong Scaling Visualization
# ============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training time vs cores
axes[0].plot(strong_df['cores'], strong_df['train_time'], 'o-', color='#2196F3', lw=2, markersize=8)
axes[0].set_xlabel('Number of Cores')
axes[0].set_ylabel('Training Time (s)')
axes[0].set_title('Strong Scaling: Training Time')
axes[0].set_xticks(CORE_CONFIGS)

# Speedup
axes[1].plot(strong_df['cores'], strong_df['speedup'], 'o-', color='#4CAF50', lw=2, markersize=8, label='Actual')
axes[1].plot(strong_df['cores'], strong_df['ideal_speedup'], '--', color='gray', lw=1.5, label='Ideal (linear)')
axes[1].set_xlabel('Number of Cores')
axes[1].set_ylabel('Speedup')
axes[1].set_title('Strong Scaling: Speedup')
axes[1].legend()
axes[1].set_xticks(CORE_CONFIGS)

# Efficiency
axes[2].bar(strong_df['cores'].astype(str), strong_df['efficiency'], color='#FF9800')
axes[2].axhline(y=1.0, color='gray', linestyle='--', lw=1)
axes[2].set_xlabel('Number of Cores')
axes[2].set_ylabel('Efficiency (Speedup / Cores)')
axes[2].set_title('Strong Scaling: Efficiency')
axes[2].set_ylim(0, 1.2)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'tableau', 'strong_scaling.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2. Weak Scaling Analysis

**Definition**: Increase data size and resources proportionally.  
**Expected**: Constant execution time if system scales perfectly.  
**Reality**: Time increases due to communication overhead.

In [ ]:
# ============================================================================
# Cell 5: Weak Scaling Test
# ============================================================================
# Proportionally increase data with cores
WEAK_CONFIGS = [
    {'cores': 1, 'fraction': 0.10},
    {'cores': 2, 'fraction': 0.20},
    {'cores': 4, 'fraction': 0.40},
    {'cores': 8, 'fraction': 0.80},
]

weak_scaling_results = []

print('WEAK SCALING TEST')
print('=' * 60)
print('Config: cores & data scale proportionally\n')

for config in WEAK_CONFIGS:
    cores = config['cores']
    frac = config['fraction']
    print(f'Testing: {cores} core(s), {frac*100:.0f}% data...')
    
    try:
        SparkSession.builder.getOrCreate().stop()
    except:
        pass
    time.sleep(2)
    
    spark = create_spark(cores)
    result = time_training(spark, PARQUET_PATH, fraction=frac)
    result['cores'] = cores
    result['data_fraction'] = frac
    weak_scaling_results.append(result)
    
    print(f'  Rows: {result["rows"]:,}, Train: {result["train_time"]}s, AUC: {result["auc"]}')
    
    spark.stop()
    gc.collect()

weak_df = pd.DataFrame(weak_scaling_results)

# Normalized time (relative to baseline)
baseline_weak = weak_df.iloc[0]['train_time']
weak_df['normalized_time'] = weak_df['train_time'] / baseline_weak

print('\nWeak Scaling Results:')
print(weak_df[['cores', 'data_fraction', 'rows', 'train_time', 'normalized_time', 'auc']].to_string(index=False))

In [ ]:
# ============================================================================
# Cell 6: Weak Scaling Visualization
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training time
labels = [f'{c["cores"]}c/{c["fraction"]*100:.0f}%' for c in WEAK_CONFIGS]
axes[0].bar(labels, weak_df['train_time'], color='#9C27B0', edgecolor='black')
axes[0].axhline(y=baseline_weak, color='red', linestyle='--', lw=1.5, label='Ideal (constant)')
axes[0].set_xlabel('Cores / Data %')
axes[0].set_ylabel('Training Time (s)')
axes[0].set_title('Weak Scaling: Training Time')
axes[0].legend()

# Normalized time
axes[1].plot(range(len(labels)), weak_df['normalized_time'], 'o-', color='#E91E63', lw=2, markersize=8, label='Actual')
axes[1].axhline(y=1.0, color='gray', linestyle='--', lw=1.5, label='Ideal (1.0)')
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels(labels)
axes[1].set_xlabel('Cores / Data %')
axes[1].set_ylabel('Normalized Time')
axes[1].set_title('Weak Scaling: Normalized Execution Time')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'tableau', 'weak_scaling.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. Data Size Impact Analysis

In [ ]:
# ============================================================================
# Cell 7: Data Size Impact (Fixed Cores, Vary Data)
# ============================================================================
DATA_FRACTIONS = [0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
FIXED_CORES = 4

data_size_results = []

print('DATA SIZE IMPACT TEST')
print('=' * 60)
print(f'Fixed cores: {FIXED_CORES}')
print(f'Data fractions: {DATA_FRACTIONS}\n')

try:
    SparkSession.builder.getOrCreate().stop()
except:
    pass
time.sleep(2)

spark = create_spark(FIXED_CORES)

for frac in DATA_FRACTIONS:
    print(f'Testing {frac*100:.0f}% data...')
    result = time_training(spark, PARQUET_PATH, fraction=frac)
    result['data_fraction'] = frac
    data_size_results.append(result)
    print(f'  Rows: {result["rows"]:,}, Train: {result["train_time"]}s, AUC: {result["auc"]}')

spark.stop()

datasize_df = pd.DataFrame(data_size_results)
print('\nResults:')
print(datasize_df[['data_fraction', 'rows', 'train_time', 'auc']].to_string(index=False))

In [ ]:
# ============================================================================
# Cell 8: Data Size Impact Visualization
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(datasize_df['rows'], datasize_df['train_time'], 'o-', color='#2196F3', lw=2, markersize=8)
axes[0].set_xlabel('Number of Rows')
axes[0].set_ylabel('Training Time (s)')
axes[0].set_title('Training Time vs Data Size')
axes[0].ticklabel_format(style='sci', axis='x', scilimits=(0,0))

axes[1].plot(datasize_df['rows'], datasize_df['auc'], 'o-', color='#4CAF50', lw=2, markersize=8)
axes[1].set_xlabel('Number of Rows')
axes[1].set_ylabel('AUC-ROC')
axes[1].set_title('Model Quality vs Data Size')
axes[1].ticklabel_format(style='sci', axis='x', scilimits=(0,0))

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'tableau', 'data_size_impact.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Bottleneck Discussion

In [ ]:
# ============================================================================
# Cell 9: Bottleneck Analysis & Discussion
# ============================================================================
print('=' * 70)
print('SCALABILITY BOTTLENECK ANALYSIS')
print('=' * 70)
print('''
1. I/O BOTTLENECK:
   - CSV reading is sequential and slow (~{:.0f}s for full dataset)
   - Parquet format reduces I/O by ~3-5x (columnar, compressed)
   - Mitigation: Use Parquet with Snappy compression

2. CPU BOTTLENECK:
   - TF-IDF computation is CPU-intensive (5000-dim vectors)
   - Strong scaling shows {:.1f}x speedup with {}x cores (ideal: {}x)
   - Amdahl's Law limits parallelizable fraction
   - Mitigation: Reduce TF-IDF dimensions, use feature hashing

3. SHUFFLE BOTTLENECK:
   - Shuffle operations during groupBy and join are expensive
   - Network overhead in distributed shuffles
   - Mitigation: Partition data by label, use broadcast joins

4. MEMORY BOTTLENECK:
   - 18GB dataset exceeds single-machine RAM for sklearn
   - Spark manages memory via spill-to-disk and LRU caching
   - Mitigation: Increase executor memory, use off-heap storage

5. COMMUNICATION OVERHEAD:
   - Weak scaling shows {:.1f}x time increase vs ideal 1.0x
   - Serialization cost grows with data volume
   - Mitigation: Use Kryo serializer, reduce shuffle partitions
'''.format(
    strong_df[strong_df['cores']==1]['total_time'].values[0],
    strong_df['speedup'].max(),
    CORE_CONFIGS[-1], CORE_CONFIGS[-1],
    weak_df['normalized_time'].iloc[-1]
))

# Export all scaling data for Tableau
TABLEAU_DIR = os.path.join(PROJECT_ROOT, 'tableau', 'data')
strong_df.to_csv(os.path.join(TABLEAU_DIR, 'strong_scaling.csv'), index=False)
weak_df.to_csv(os.path.join(TABLEAU_DIR, 'weak_scaling.csv'), index=False)
datasize_df.to_csv(os.path.join(TABLEAU_DIR, 'data_size_impact.csv'), index=False)

print('Scaling data exported to tableau/data/')